# 01 · How does a computer *see* an image?

Look at a photo and you instantly see an airplane. A computer just sees a grid of numbers. Here we'll follow **one** tiny image through CLIP, step by step, until it turns into a list of 512 numbers that can be compared with text.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 2, Section 2.2, *Vision Transformers and Patch-Based Encoders*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

### Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 1. Which image are we following?

Models are way easier to understand when you track one concrete example all the way through. So first things first: let's pick an image.

### 1.1 Grab our tools

In [ ]:
import matplotlib.pyplot as plt  # plotting, arrays, tensors, datasets, and CLIP
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise

### 1.2 Where do the images come from?

We'll use [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html), a classic dataset of tiny 32 × 32 color photos in ten categories: airplanes, cars, birds, cats, and so on. They're so small you can almost count the pixels, which is exactly what we want. Nothing gets hidden.

<img src="../assets/diagrams/notebook1/cifar10.png" alt="Ten example categories in CIFAR-10" width="800">

In [ ]:
DATASET_ID = "uoft-cs/cifar10"  # CIFAR-10 on the Hugging Face Hub
DATASET_REVISION = "0b2714987fa478483af9968de7c934580d0bb9a2"  # pin the exact version, so row 0 is the same image for everyone

And who's doing the looking? **CLIP ViT-B/32**. The name looks like alphabet soup, but it actually tells you a lot:

- **ViT**: it's a *Vision Transformer*.
- **B**: it's the *Base* size. Bigger versions exist.
- **/32**: it cuts images into **32 × 32-pixel patches**. Keep this number in your back pocket; it shows up again in Section 4.

<img src="../assets/diagrams/notebook1/clip-base-p32.png" alt="CLIP-B/32 vision and text encoders" width="800">

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"  # one name gets us both the preprocessing recipe and the weights

Time to load the data. The first run downloads CIFAR-10; after that it just reads from your local cache, so it's quick.

We also pass a `revision`, which works like a commit hash for the dataset. That way row 0 is the same airplane for everyone, even if the dataset gets updated later.

In [ ]:
train_data = load_dataset(  # downloads once, then loads from the cache
    DATASET_ID,
    "plain_text",
    split="train",
    revision=DATASET_REVISION,
)

So what's actually in each record? Let's ask the dataset for its **features**, which is just a fancy word for its columns:

In [ ]:
print(train_data.features)  # what columns does each record have?

Two columns:

- `img`: the picture itself, as a PIL image
- `label`: the category, stored as a plain integer. The `ClassLabel` type also remembers the ten names that go with those integers.

### 1.3 Pick our running example

Let's just take the very first image. It comes back as a **PIL image**, which is Python's go-to image object. Nice for displaying, but it's not a PyTorch tensor yet. We'll get there.

In [ ]:
record = train_data[0]  # the very first image in the training set
pil_image = record["img"]

The label is just a number, which isn't very readable. `int2str` looks up the matching class name for us:

In [ ]:
label_id = record["label"]  # an integer from 0 to 9...
label_name = train_data.features["label"].int2str(label_id)  # ...turned back into its class name

Quick sanity check before we start poking at it. You should see `airplane`, a size of 32 × 32, and color mode `RGB`.

In [ ]:
print("label:", label_name, "| size [W, H]:", pil_image.size, "| mode:", pil_image.mode)  # quick sanity check

## 2. What does an image look like to a computer?

To us it's a picture. To the computer it's a block of numbers. Let's stop pretending and look at the numbers.

### 2.1 Open up the pixel array

A color image stores **three numbers per pixel**: how much red, green, and blue light is at that spot. Turning the image into a NumPy array shows us exactly that, no extras.

In [ ]:
pixel_array = np.asarray(pil_image).copy()  # .copy() gives us our own writable array

When you run the next cell, look for three things:

1. **Shape**: `(32, 32, 3)` means 32 rows (height), 32 columns (width), and 3 color channels.
2. **Data type**: `uint8`, an unsigned 8-bit integer. So every value is a whole number from 0 to 255.
3. **Range**: the smallest and biggest values *in this particular image*. Don't expect exactly 0 and 255.

In [ ]:
print("array [H, W, C]:", pixel_array.shape, "| dtype:", pixel_array.dtype)  # HWC: height, width, channels
print("byte range:", pixel_array.min(), "to", pixel_array.max())

**TODO:** Let's peek at a single pixel. Change `row` and `col` (anything from 0 to 31) and see what you get.

In [ ]:
# TODO: try a few spots, like a corner vs. the middle of the image
row, col = 0, 0
print(f"pixel ({row}, {col}) -> R, G, B =", pixel_array[row, col])

Three numbers, one per color. That's all a pixel is. Can you find a spot where one color clearly wins?

### 2.2 Why does PyTorch want the channels first?

Here's a little gotcha. Image libraries usually store pixels as **height × width × channels** (people write it **HWC**). PyTorch vision models want **channels × height × width** (**CHW**). Same numbers, just shuffled around.

Two moves get us there:

- `permute(2, 0, 1)` takes the channel axis (position 2) and moves it to the front.
- `/ 255.0` squishes the bytes from 0–255 down to decimals between 0 and 1.

> **Heads up:** We're only building this tensor to understand the layout. CLIP has its own preprocessing, and that's what we'll actually use in Section 3.

In [ ]:
raw_hwc = torch.from_numpy(pixel_array)
raw_chw = raw_hwc.permute(2, 0, 1).float() / 255.0  # just for learning the layout; CLIP uses its own processor

> **Pause and predict:** The HWC shape is `(32, 32, 3)`. What's the CHW shape going to be? Which axis moved?

Run it and see if you were right.

In [ ]:
print("raw HWC:", tuple(raw_hwc.shape))  # same pixels, two layouts
print("raw CHW:", tuple(raw_chw.shape))

## 3. How do we get the image ready for CLIP?

CLIP never saw raw 32 × 32 images during training. Every image it learned from was resized, cropped, and color-normalized in one very specific way. If we want CLIP to make sense of our image, we have to prep it **exactly the same way**.

### 3.1 Load the processor and the model

Hugging Face splits this into two pieces:

- `CLIPProcessor` knows the **prep recipe**: resize, crop, normalize.
- `CLIPModel` holds the **learned weights**.

Always load both from the same checkpoint name. The recipe belongs to the model, and if you mix them up, nothing crashes. You just quietly get worse results, which is the worst kind of bug.

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)  # the prep recipe that belongs to this checkpoint

Now the model. We move it to our device and call `.eval()`. We're not training anything today, and eval mode turns off training-only stuff like dropout, so we get the same answer every time we run it.

In [ ]:
model = CLIPModel.from_pretrained(MODEL_ID).to(device)  # the pretrained weights
model.eval()  # we're only looking, not training: turn off dropout

No need to dig up an architecture table. We can just ask the model about itself. Two numbers matter for the rest of this notebook:

- **patch size**: how many pixels wide each image patch is
- **vision width**: how many numbers the transformer uses to describe each patch

In [ ]:
print("patch size:", model.config.vision_config.patch_size)  # the model knows its own architecture
print("vision width:", model.config.vision_config.hidden_size)

Patch size 32, just like the name promised. Vision width 768.

One more number to know. CLIP also has a text side, and to compare an image with a sentence, both get mapped into a **shared space** of the same size. How big is it?

In [ ]:
print("shared projection width:", model.config.projection_dim)  # where image and text vectors meet

512. These two widths are super easy to mix up, so here they are side by side:

| Width | Where it lives |
|---|---|
| **768** | inside the vision transformer |
| **512** | the shared space, where images and text get compared |

### 3.2 What does CLIP actually receive?

Let's run the processor on our image. One thing to know: it always hands back a **batch**, even if you only gave it one image. The part we care about is `pixel_values`. That's exactly what the vision model will read.

In [ ]:
model_inputs = processor(images=pil_image, return_tensors="pt").to(device)  # resize, crop, normalize
pixel_values = model_inputs["pixel_values"]  # the tensor the vision model actually reads

> **Pause and predict:** Our image is 32 × 32 pixels. What shape do you think the processor will give back?

In [ ]:
print("CLIP pixels [B, C, H, W]:", tuple(pixel_values.shape))  # note the batch dimension in front
print("normalized range:", pixel_values.min().item(), "to", pixel_values.max().item())

`[1, 3, 224, 224]`: a batch of **1** image, **3** color channels, **224 × 224** pixels. Yep, the processor blew our tiny image *up* to the size CLIP was trained on.

Now check the range. Some values are negative, some are above 1. What happened? Normalization subtracts a mean and divides by a standard deviation, separately for each color channel. So these are numbers for the model, not colors for your screen.

**So how do we see what CLIP sees?** We undo the normalization on a separate copy. The processor keeps one mean and one standard deviation per color channel. If we reshape them to `[1, 3, 1, 1]`, PyTorch will apply them to every pixel at once. (That trick is called *broadcasting*.)

In [ ]:
# Display only: one mean and one std per color channel, shaped so they broadcast over every pixel
image_mean = torch.tensor(processor.image_processor.image_mean, device=device).view(1, 3, 1, 1)
image_std = torch.tensor(processor.image_processor.image_std, device=device).view(1, 3, 1, 1)

Undoing normalization is just the reverse: multiply by the std, add the mean back. Then we flip the channels back to the end (HWC), because that's what Matplotlib likes. The original `pixel_values` doesn't get touched, so the model still gets the right input.

In [ ]:
restored_pixels = (pixel_values * image_std + image_mean).clamp(0, 1)  # undo normalization on a copy
processed_for_plot = restored_pixels[0].permute(1, 2, 0).cpu()  # CHW -> HWC for Matplotlib

Here it is: the image CLIP actually works with. From now on, everything happens to this 224 × 224 version, not the original 32 × 32 one.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))  # the 224 × 224 crop that CLIP sees
ax.imshow(processed_for_plot)
ax.set_title("CLIP's 224 × 224 image input")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. How does CLIP cut the image into pieces?

Transformers read a **sequence of tokens**, like words in a sentence. But an image is a 2D grid, not a sentence. So what do we do?

The Vision Transformer's trick is surprisingly simple: **cut the image into square patches, and treat each patch like a word.**

This idea comes from the Vision Transformer paper, [*An Image is Worth 16x16 Words*](https://arxiv.org/abs/2010.11929) (Dosovitskiy et al., 2020). Its Figure 1 is pretty much a map of the rest of this notebook:

<img src="../assets/diagrams/notebook1/vit_fig1.png" alt="Vision Transformer model overview" width="800">

*Figure 1 from Dosovitskiy et al. (2020).*

Read it from the bottom up:

1. **The image gets cut into patches** (bottom left). The paper's example uses 16 × 16 patches; our CLIP-B/32 uses 32 × 32. We do this in Section 4.1.
2. **Linear Projection of Flattened Patches** turns each patch into a vector. In CLIP, that's the patch convolution, also in Section 4.1.
3. **Patch + Position Embedding:** each patch vector gets a position added, so the model knows where it came from. The extra token marked `0*` is the class token. CLIP adds both inside `model.vision_model`, and we'll meet the class token in Section 5.1.
4. **Transformer Encoder** lets all the positions share information (Section 5.1).
5. **MLP Head** turns the class token into a class prediction. This is where CLIP does something different: instead of predicting a class, it projects the class token into the shared image–text space (Section 5.2).

### 4.1 From pixels to patch tokens

CLIP-B/32 does the cutting with a single convolution layer:

- Its **kernel is 32 × 32**, so it looks at one patch at a time.
- Its **stride is 32**, so it jumps a whole patch each step. No overlap.

That one layer pulls double duty. It **cuts** the image into patches and **turns** each patch into 768 learned numbers.

In [ ]:
with torch.inference_mode():  # no gradients needed, we're just reading features
    patch_grid = model.vision_model.embeddings.patch_embedding(pixel_values)

> **Pause and predict:** The image is 224 pixels wide, and each patch is 32 pixels wide. How many patches fit along each side?

The output comes back as `[batch, features, rows, columns]`.

In [ ]:
grid_height, grid_width = patch_grid.shape[-2:]  # read the grid size instead of hard-coding 7
print("patch grid [B, D, rows, cols]:", tuple(patch_grid.shape))

224 ÷ 32 = 7, so we get a **7 × 7 grid**, and every cell holds 768 numbers.

But a transformer wants a flat line of tokens, not a grid. Two moves fix that:

- `flatten(2)` squashes rows and columns into one axis: 7 × 7 → 49 positions.
- `transpose(1, 2)` swaps axes so each position comes first, followed by its 768 features.

We should end up with `[1, 49, 768]`: 49 patch tokens, 768 numbers each.

In [ ]:
patch_tokens = patch_grid.flatten(2).transpose(1, 2)  # [1, 768, 7, 7] -> [1, 49, 768]
print("learned patch tokens [B, N, D]:", tuple(patch_tokens.shape))

Let's draw those 49 patches right on the image. Each square turns into one token, read left to right, top to bottom, like a page in a book.

> **Something to chew on:** We've got 49 patch tokens. But in Section 5, the transformer is going to process **50** positions. Where does the extra one come from? *Hint: something has to summarize the whole image.*

In [ ]:
patch_size = model.config.vision_config.patch_size
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(processed_for_plot)
for boundary in range(patch_size, pixel_values.shape[-1], patch_size):  # a line at every patch edge
    ax.axhline(boundary - 0.5, color="white", linewidth=1)
    ax.axvline(boundary - 0.5, color="white", linewidth=1)
ax.set_title("CLIP's learned 7 × 7 patch grid")
ax.axis("off")
plt.tight_layout()
plt.show()

### 4.2 Zoom in on a single patch

Let's grab one patch, say row 2, column 3 (counting from zero), and cut out its 32 × 32 pixels. At the start, this little square is *everything* that one patch token gets to see.

**TODO:** Once you've seen this one, change `patch_row` and `patch_column` (each from 0 to 6) and re-run the next two cells.

In [ ]:
patch_row, patch_column = 2, 3  # TODO: try other patches (0 to 6 each)
chosen_patch = processed_for_plot[
    patch_row * patch_size:(patch_row + 1) * patch_size,
    patch_column * patch_size:(patch_column + 1) * patch_size,
]  # slice its 32 × 32 pixels using the model's own patch edges

Can you tell what this is? Probably not, and that's totally fine. On its own, one patch is just a small scrap of local evidence. The fun part comes next, when the transformer lets all the patches talk to each other.

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(chosen_patch, interpolation="nearest")  # "nearest" keeps pixels blocky so you can actually see them
ax.set_title(f"Processed patch ({patch_row}, {patch_column})")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. How do 49 patches become *one* vector?

We've got 49 separate patch tokens now. But to compare an image with a sentence, we need **one** vector for the whole image. How does CLIP get there?

### 5.1 Let the patches talk to each other

Remember that extra 50th position? Here's the answer: CLIP sticks one extra learned token, the **class token** (you'll often see it written `[CLS]`), in front of the 49 patches. 49 + 1 = 50.

Then the transformer runs **self-attention**, which lets every position look at every other position. So a patch of plain sky can now "know" there's a wing in the patch next to it.

In [ ]:
with torch.inference_mode():  # run the whole vision transformer
    vision_outputs = model.vision_model(pixel_values=pixel_values)

The model hands back a few different views of its result. We'll keep two:

- `last_hidden_state`: all 50 positions, after they've shared context
- `pooler_output`: the class token's final state, after one last layer norm. That's CLIP's summary of the whole image.

In [ ]:
final_states = vision_outputs.last_hidden_state  # all 50 positions, after attention
pooled_state = vision_outputs.pooler_output  # the class token's summary of the whole image

Check the shapes: `[1, 50, 768]` for everything, `[1, 768]` for the summary.

And just to be clear, the summary is *not* an average of pixels. Through attention, the class token has pulled in information from every single patch. It's a *learned* summary.

In [ ]:
print("final states [B, 1+N, D]:", tuple(final_states.shape))  # attention changes values, not the number of positions
print("pooled vision state [B, D]:", tuple(pooled_state.shape))

### 5.2 Step into the shared image–text space

Our summary still lives in the vision tower's 768-number world. Two last steps get it ready to meet text:

1. **Project:** a learned linear layer maps 768 numbers down to 512, the same size as CLIP's text vectors.
2. **Normalize:** scale the vector so its length is exactly 1 (this is called *L2 normalization*).

In [ ]:
with torch.inference_mode():
    projected_image = model.visual_projection(pooled_state)  # CLIP's learned 768 -> 512 projection
image_vector = F.normalize(projected_image, dim=-1)  # length 1, so dot product = cosine similarity

You should see `[1, 512]` and a length super close to `1.0`.

**Why bother making it length 1?** Because for two length-1 vectors, the dot product *is* their **cosine similarity**: a score from −1 to 1 that tells you how much they point the same way. That's exactly how CLIP compares images with text, and we'll do it ourselves in Notebook 03.

In [ ]:
print("normalized image vector [B, D]:", tuple(image_vector.shape))  # one row per image
print("vector length:", image_vector.norm(dim=-1))

And finally, here are all 512 numbers as a strip of colors.

> **Heads up:** Don't go hunting for an "airplane number" or a "sky number" in there. The meaning is smeared across the whole pattern. That's why we always compare whole vectors, never single numbers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))  # all 512 numbers in one row
heatmap = ax.imshow(image_vector.cpu(), aspect="auto", cmap="coolwarm")
ax.set_xlabel("shared image-text coordinate")
ax.set_yticks([0], ["image 0"])
ax.set_title("Normalized CLIP image vector")
fig.colorbar(heatmap, ax=ax, label="value")
plt.tight_layout()
plt.show()

## Recap

That's the whole trip. Our one CIFAR-10 image changed form four times:

```text
32 × 32 RGB pixels                         the raw image
      ↓ CLIP's processor: resize, crop, normalize
[1, 3, 224, 224] model input
      ↓ 32 × 32 patch convolution
49 patch tokens + 1 class token            [1, 50, 768]
      ↓ transformer -> class token -> projection -> normalize
one unit-length vector                     [1, 512]
```

**The big ideas:**

- An image is just a grid of numbers, and the **layout** (HWC vs. CHW) matters as much as the values.
- Always prep images with the processor that matches your checkpoint.
- A Vision Transformer treats **image patches like words**.
- Self-attention lets patches share context, and the **class token** gathers it all into one summary.
- The final vector has length 1, so a plain dot product tells you how similar two things are.

### Check your understanding

1. Why can't we just feed `raw_chw` from Section 2.2 straight into CLIP?
2. Say CLIP used 16 × 16 patches (like ViT-B/16 does) on the same 224 × 224 input. How many patch tokens would we get?
3. Why do we bother normalizing the final vector to length 1?

<details>
<summary>Show answers</summary>

1. It skips CLIP's prep. It's the wrong size (32 × 32 instead of 224 × 224), and it isn't normalized with the mean and std CLIP was trained with.
2. 224 ÷ 16 = 14 patches per side, so 14 × 14 = **196** patch tokens (197 positions once you add the class token).
3. With length 1, the dot product of two vectors equals their cosine similarity. That compares *direction* without length messing things up.

</details>

## Try it yourself

### TODO 1: Run a different image through the whole pipeline

The cell below does everything from this notebook in a few lines: prep, vision transformer, projection, normalize. It runs as-is, but the fun part is changing `other_index`.

Before you run it, guess: will the shape change? Will the similarity to our airplane be high or low?

In [ ]:
# TODO: change other_index (anything from 0 to 49,999) and re-run
other_index = 1234
other_record = train_data[other_index]
other_label = train_data.features["label"].int2str(other_record["label"])

with torch.inference_mode():
    other_pixels = processor(images=other_record["img"], return_tensors="pt")["pixel_values"].to(device)
    other_pooled = model.vision_model(pixel_values=other_pixels).pooler_output
    other_vector = F.normalize(model.visual_projection(other_pooled), dim=-1)

print("label:", other_label, "| vector shape:", tuple(other_vector.shape))
print("cosine similarity with our airplane:", round((image_vector @ other_vector.T).item(), 3))

The shape never changes: it's always `[1, 512]`, no matter what's in the picture. Only the numbers do.

### TODO 2: Does CLIP think airplanes look alike?

Use the cell above to hunt down **another airplane** and something very different, like a **frog**. Which one scores higher against our airplane?

<details>
<summary>Hint</summary>

You don't have to guess indices. This finds the first few images of a given class:

```python
frog_id = train_data.features["label"].str2int("frog")
[i for i, label in enumerate(train_data["label"][:200]) if label == frog_id][:5]
```

You'd usually expect the other airplane to score higher, but not always. Backgrounds and colors count too. That's a good hint that "similar vectors" isn't the same thing as "same category". We'll dig into this in Notebooks 03 and 05.

</details>

### TODO 3: Break the prep on purpose

Remember how mixing up the prep recipe "quietly gives you worse results"? Try it. In TODO 1, feed the vision model the image **without** normalization: build `other_pixels` with `processor(..., do_normalize=False)` instead. How much does the similarity move?

---

**Up next, Notebook 02:** how does a *sentence* turn into a vector? We'll follow the text side of CLIP, starting with tokenization.